In [ ]:
!pip install jams

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 5.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

jams_filename = '/content/drive/MyDrive/Capstone/Tabs/00_BN1-129-Eb_comp.jams'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [ ]:
import jams

jam = jams.load(jams_filename)

print(f"File version: {jam.file_metadata.jams_version}")
print(f"Duration: {jam.file_metadata.duration:.2f} seconds")
print(f"Title: {jam.file_metadata.title}")
print(f"Artist: {jam.file_metadata.artist}")
print(f"\nTotal annotations: {len(jam.annotations)}")
print(f"\nAnnotation namespaces present:")
for i, anno in enumerate(jam.annotations):
    print(f"  [{i}] {anno.namespace}  ({len(anno.data)} observations)")

File version: 0.3.1
Duration: 22.32 seconds
Title: 00_BN1-129-Eb_comp
Artist: 

Total annotations: 17

Annotation namespaces present:
  [0] pitch_contour  (710 observations)
  [1] note_midi  (7 observations)
  [2] pitch_contour  (1964 observations)
  [3] note_midi  (23 observations)
  [4] pitch_contour  (2164 observations)
  [5] note_midi  (33 observations)
  [6] pitch_contour  (2342 observations)
  [7] note_midi  (36 observations)
  [8] pitch_contour  (1648 observations)
  [9] note_midi  (25 observations)
  [10] pitch_contour  (878 observations)
  [11] note_midi  (9 observations)
  [12] beat_position  (48 observations)
  [13] tempo  (1 observations)
  [14] chord  (6 observations)
  [15] chord  (6 observations)
  [16] key_mode  (1 observations)


In [ ]:
# Key
key_annotations = jam.search(namespace='key_mode')
if key_annotations:
    key_obs = key_annotations[0].data[0]
    print(f"Key: {key_obs.value}")
    print(f"  (from {key_obs.time:.2f}s, duration {key_obs.duration:.2f}s)")

# Tempo / beats
beat_annotations = jam.search(namespace='beat_position')
if beat_annotations:
    beats = beat_annotations[0].data
    print(f"\nBeats: {len(beats)} beat markers")
    print(f"First 5 beats at: {[f'{b.time:.2f}s' for b in beats[:5]]}")

# Chords
chord_annotations = jam.search(namespace='chord')
if chord_annotations:
    chords = chord_annotations[0].data
    print(f"\nChord progression: {len(chords)} chord segments")
    for ch in chords[:10]:
        print(f"  {ch.time:5.2f}s  ({ch.duration:.2f}s)  {ch.value}")

Key: Eb:major
  (from 0.00s, duration 22.32s)

Beats: 48 beat markers
First 5 beats at: ['0.00s', '0.47s', '0.93s', '1.40s', '1.86s']

Chord progression: 6 chord segments
   0.00s  (7.44s)  D#:maj
   7.44s  (3.72s)  G#:maj
  11.16s  (3.72s)  D#:maj
  14.88s  (1.86s)  A#:maj
  16.74s  (1.86s)  G#:maj
  18.60s  (3.72s)  D#:maj


In [ ]:
# String index 0 = low E (thickest, lowest pitch)
# String index 5 = high E (thinnest, highest pitch)
string_names = ['Low E (6)', 'A (5)', 'D (4)', 'G (3)', 'B (2)', 'High E (1)']

note_annotations = jam.search(namespace='note_midi')
print(f"Found {len(note_annotations)} note tracks (one per string)\n")

for string_idx, anno in enumerate(note_annotations):
    notes = anno.data
    print(f"String {string_idx} — {string_names[string_idx]}: {len(notes)} notes")
    for obs in notes[:5]:
        # obs.value is the MIDI pitch
        print(f"    start={obs.time:5.2f}s  dur={obs.duration:.2f}s  midi={obs.value:.1f}")
    print()

Found 6 note tracks (one per string)

String 0 — Low E (6): 7 notes
    start= 7.46s  dur=0.46s  midi=44.0
    start= 7.92s  dur=0.92s  midi=44.2
    start= 8.87s  dur=0.19s  midi=44.1
    start=14.83s  dur=0.49s  midi=46.1
    start=15.33s  dur=1.42s  midi=46.1

String 1 — A (5): 23 notes
    start= 0.05s  dur=0.42s  midi=51.0
    start= 0.52s  dur=0.42s  midi=51.0
    start= 0.94s  dur=0.87s  midi=51.0
    start= 1.89s  dur=0.45s  midi=51.0
    start= 2.34s  dur=0.23s  midi=51.0

String 2 — D (4): 33 notes
    start= 0.72s  dur=0.86s  midi=58.2
    start= 1.63s  dur=0.76s  midi=58.2
    start= 3.07s  dur=0.59s  midi=58.1
    start= 3.73s  dur=0.87s  midi=58.2
    start= 4.66s  dur=1.17s  midi=58.1

String 3 — G (3): 36 notes
    start= 0.05s  dur=0.46s  midi=62.1
    start= 1.18s  dur=0.57s  midi=62.0
    start= 1.90s  dur=0.15s  midi=62.1
    start= 2.08s  dur=0.34s  midi=62.1
    start= 2.60s  dur=0.42s  midi=62.1

String 4 — B (2): 25 notes
    start= 0.05s  dur=0.45s  midi=65.0
 

In [ ]:
# Standard guitar tuning — open string MIDI values
# Index 0 (low E) = 40, Index 5 (high E) = 64
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]

def midi_to_note_name(midi):
    notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    return f"{notes[int(midi) % 12]}{int(midi) // 12 - 1}"

print("Ground truth notes with derived fret positions:\n")
all_notes = []
for string_idx, anno in enumerate(note_annotations):
    for obs in anno.data:
        midi_pitch = obs.value
        fret = round(midi_pitch - OPEN_STRING_MIDI[string_idx])
        all_notes.append({
            'start': obs.time,
            'duration': obs.duration,
            'string': string_idx,
            'fret': fret,
            'midi': round(midi_pitch),
            'note_name': midi_to_note_name(midi_pitch)
        })

# Sort by start time (chronological)
all_notes.sort(key=lambda n: n['start'])

# Show first 20
print(f"{'Time':>7}  {'Dur':>5}  {'String':>6}  {'Fret':>4}  {'Note':>4}")
print("-" * 40)
for n in all_notes[:20]:
    print(f"{n['start']:6.2f}s  {n['duration']:.2f}s  {n['string']:>6}  {n['fret']:>4}  {n['note_name']:>4}")

print(f"\nTotal notes in recording: {len(all_notes)}")

Ground truth notes with derived fret positions:

   Time    Dur  String  Fret  Note
----------------------------------------
  0.05s  0.42s       1     6   D#3
  0.05s  0.45s       4     6    F4
  0.05s  0.46s       3     7    D4
  0.52s  0.42s       1     6   D#3
  0.72s  0.86s       2     8   A#3
  0.94s  0.87s       1     6   D#3
  1.18s  0.45s       4     8    G4
  1.18s  0.57s       3     7    D4
  1.63s  0.76s       2     8   A#3
  1.89s  0.45s       1     6   D#3
  1.89s  0.51s       4     6    F4
  1.90s  0.15s       3     7    D4
  2.08s  0.34s       3     7    D4
  2.34s  0.23s       1     6   D#3
  2.60s  0.71s       4     8    G4
  2.60s  0.42s       3     7    D4
  2.60s  0.65s       1     6   D#3
  3.03s  0.32s       3     7    D4
  3.07s  0.59s       2     8   A#3
  3.30s  0.87s       1     6   D#3

Total notes in recording: 133


In [ ]:
from collections import defaultdict

# Group notes that start at the same time (within 50ms) into chord events
chord_events = defaultdict(list)
TIME_TOLERANCE = 0.05  # 50ms

sorted_notes = sorted(all_notes, key=lambda n: n['start'])
current_group_time = None
current_group = []

groups = []
for note in sorted_notes:
    if current_group_time is None or note['start'] - current_group_time > TIME_TOLERANCE:
        if current_group:
            groups.append((current_group_time, current_group))
        current_group_time = note['start']
        current_group = [note]
    else:
        current_group.append(note)
if current_group:
    groups.append((current_group_time, current_group))

print(f"Found {len(groups)} chord/note events (grouped by simultaneous onsets)\n")
print(f"First 10 events:")
for start_time, group in groups[:10]:
    note_summary = ', '.join(f"{n['note_name']}(s{n['string']}f{n['fret']})" for n in group)
    print(f"  {start_time:5.2f}s  ({len(group)} notes): {note_summary}")

Found 75 chord/note events (grouped by simultaneous onsets)

First 10 events:
   0.05s  (3 notes): D#3(s1f6), F4(s4f6), D4(s3f7)
   0.52s  (1 notes): D#3(s1f6)
   0.72s  (1 notes): A#3(s2f8)
   0.94s  (1 notes): D#3(s1f6)
   1.18s  (2 notes): G4(s4f8), D4(s3f7)
   1.63s  (1 notes): A#3(s2f8)
   1.89s  (3 notes): D#3(s1f6), F4(s4f6), D4(s3f7)
   2.08s  (1 notes): D4(s3f7)
   2.34s  (1 notes): D#3(s1f6)
   2.60s  (3 notes): G4(s4f8), D4(s3f7), D#3(s1f6)
